In [ ]:
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
CODE, URL = '/content/code', 'github.com/quocthangtrann/computer_vision_ADAS.git'
env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
if not os.path.exists(CODE):
    subprocess.run(f'git clone https://{URL} {CODE}', shell=True, check=True, env=env)
else:
    subprocess.run(f'git -C {CODE} pull', shell=True, check=True, env=env)
subprocess.run(f'rsync -a --exclude .git {CODE}/repo/ /content/drive/MyDrive/capstone_ai/repo/', shell=True, check=True)
try: import segmentation_models_pytorch
except ImportError: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation-models-pytorch'], check=True)
os.chdir('/content/drive/MyDrive/capstone_ai')
print('mã mới nhất đã ở capstone_ai/repo/ | thư mục làm việc:', os.getcwd())


In [ ]:
import subprocess, sys, time, os
import torch; assert torch.cuda.is_available(), 'Chưa có GPU: đổi runtime sang GPU'
B9, TAG, SEEDS, EPOCHS = 'shard_000', 'b9s1', range(5), 60    # 2 shard: B9='shard_000,shard_001', TAG='b9s2'
T = time.time()
for s in SEEDS:
    n = f'{TAG}_c4aug_s{s}'
    if os.path.exists(f'runs/{n}/final_metrics.json'): print(n, 'đã xong, bỏ qua'); continue
    print(f'\n>>> {n}  [{(time.time() - T) / 60:.1f} phút]', flush=True)
    pr = subprocess.Popen([sys.executable, '-u', 'repo/train_v3.py', '--name', n, '--seed', str(s), '--b9', B9, '--epochs', str(EPOCHS)],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with open(f'runs/{n}.log', 'a') as lg:
        for ln in pr.stdout:
            lg.write(ln)
            if not ln.startswith('ep ') or ln.split()[1].split('/')[0] in ('1', '10', '20', '30', '40', '50', '60'): print(ln, end='', flush=True)
    rc = pr.wait(); print('mã thoát:', rc, flush=True)
    if rc != 0: print('LỖI: dừng lại, dán đoạn trên cho tôi'); break
else: print(f'\nXONG sau {(time.time() - T) / 60:.1f} phút')
